In [0]:
import logging
import pyspark.sql.functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

In [0]:
catalog_name = "supply_chain_daily_demand"
bronze_schema_name = "bronze"
raw_schema_name = "raw"
volume_name = "data"
bronze_table_name = "raw_daily_demand"
file_name = "supply_chain_dataset.csv"

In [0]:
class Bronze:
    def load_raw_data(self):
        file_path = f"/Volumes/{catalog_name}/{raw_schema_name}/{volume_name}/{file_name}"

        logging.info(f"File loading from the file path: {file_path}.")
        self.data = spark.read.format("csv")\
                              .option("header", "true")\
                              .option("sep", ",")\
                              .option("inferSchema", "true")\
                              .load(file_path)

        logging.info(f"File loaded and saved in the dataframe.")
        #self.data.show()

    def write_to_bronze(self):
        bronze_table_path = f"{catalog_name}.{bronze_schema_name}.{bronze_table_name}"

        logging.info("Dropping columns: ingestion_timestamp, source_system, operation_type")
        self.data = self.data.drop("ingestion_timestamp", "source_system", "operation_type")

        logging.info("Adding the ingesting timestamp columns: bronze_ingestion_timestamp")
        self.data = self.data.withColumn("bronze_ingestion_timestamp", F.current_timestamp())

        logging.info(f"Writing the loaded dataframe to delta table, table_path: {bronze_table_path}.")
        self.data.write.format("delta")\
                       .mode("overwrite")\
                       .saveAsTable(bronze_table_path)
        logging.info(f"Data saved in delta table, path: {bronze_table_path}")

In [0]:
bronze = Bronze()
bronze.load_raw_data()
bronze.write_to_bronze()

In [0]:
%sql
-- SELECT * FROM supply_chain_daily_demand.bronze.raw_daily_demand;